# Sommelier Kaggle - Original Stage 1 (Unmodified for A/B Testing)

Notebook này là bản single-file để chạy full pipeline trên Kaggle:
- Tự clone repo.
- Tự cài dependencies từ internet.
- Stage 3 dùng `ClearVoice` + `MossFormer2_SS_16K` self-host/local để tách speech overlap, không gọi Hugging Face Inference API.
- Stage 4 chạy Whisper large-v3 + PhoWhisper local + ChunkFormer local, không dùng PhoWhisper API.
- Không phụ thuộc notebook `00_v00_build_wheels_dataset.ipynb` hay private wheels dataset.

Điều kiện trước khi chạy:
- Kaggle Accelerator: GPU bật cho các stage nặng.
- Kaggle Internet: bật để cài dependencies và tải model local lần đầu.
- Kaggle Secret có `HF_TOKEN` cho các model gated như pyannote.
- Dataset audio đã Add Input vào notebook.

Lưu ý: bản này tối ưu cho podcast. Stage 3 mark overlap từ 0.05s và thử MossFormer từ 0.10s để giữ các backchannel ngắn như “vâng”, “ừ”, “dạ”, “đúng rồi”. Các đoạn micro overlap vẫn được gắn nhãn review để không đưa nhầm vào clean train.

**Mục đích:** Chạy độc lập Stage 1 nguyên gốc từ bản V4 để so sánh đối chứng (A/B testing) với bản đã tinh chỉnh tham số.


## 0. Cấu hình run

Chỉnh các biến bên dưới nếu muốn đổi branch, giới hạn thời lượng test, hoặc tắt bước nặng.

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "test-divide-stage"

RUN_DIR = "/kaggle/working/run_full"
INPUT_DIR = f"{RUN_DIR}/00_input"
DIAR_DIR = f"{RUN_DIR}/01_diarization"
MUSIC_DIR = f"{RUN_DIR}/02_music_clean"
OVERLAP_DIR = f"{RUN_DIR}/03_overlap"
ASR_DIR = f"{RUN_DIR}/04_asr"
EXPORT_DIR = f"{RUN_DIR}/05_export"
FINAL_DIR = f"{EXPORT_DIR}/final"
EVAL_DIR = f"{RUN_DIR}/06_eval"
PREVIEW_DIR = f"{RUN_DIR}/preview"
LOG_DIR_PATH = f"{RUN_DIR}/logs"
AUDIO_WAV = f"{INPUT_DIR}/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = 300

# Bước nặng. Có thể tắt để debug nhanh.
RUN_DEMUCS = True

# Stage 01 podcast cleanup: chỉ merge cùng speaker khi gap rất nhỏ, giữ backchannel khác speaker.
SAME_SPEAKER_MERGE_GAP_SECONDS = 0.30
SHORT_BACKCHANNEL_SECONDS = 1.0

# Full-duplex train grouping: chọn 2 speaker chính rồi chia clean/review/exclude.
EXPECTED_MAIN_SPEAKERS = 2
MAIN_SPEAKERS = []  # Ví dụ ["SPEAKER_04", "SPEAKER_05"] nếu muốn ép 2 người chính thủ công.
EXPORT_PARTITION_BY_DUPLEX_GROUP = True

# Stage 03 dùng ClearVoice/MossFormer2 self-host để tách overlap, không gọi API.
RUN_OVERLAP_MARK_ONLY = False
RUN_MOSSFORMER_SEPARATION = True
MOSSFORMER_MODEL_NAME = "MossFormer2_SS_16K"
MOSSFORMER_TASK = "speech_separation"
MOSSFORMER_FAIL_OPEN = True
MOSSFORMER_MIN_OVERLAP_SECONDS = 0.10
MOSSFORMER_MIN_SEGMENT_SECONDS = 0.15
MOSSFORMER_CONTEXT_SECONDS = 0.8
MOSSFORMER_MAX_WINDOW_SECONDS = 6.0
MOSSFORMER_SAVE_WINDOW_STEMS = True

# Speaker-safe stem assignment: dùng embedding để map stem -> speaker; low-confidence không đưa sang ASR.
MOSSFORMER_USE_SPEAKER_EMBEDDING_ASSIGNMENT = True
MOSSFORMER_EMBEDDING_DEVICE = "cpu"  # CPU tránh chiếm thêm VRAM khi ClearVoice đang chạy GPU.
MOSSFORMER_EMBEDDING_FAIL_OPEN = True
MOSSFORMER_REFERENCE_MIN_SECONDS = 1.5
MOSSFORMER_REFERENCE_MAX_SEGMENTS_PER_SPEAKER = 6
MOSSFORMER_ASSIGNMENT_MIN_CONFIDENCE = 0.55
MOSSFORMER_ASSIGNMENT_MIN_MARGIN = 0.08
MOSSFORMER_REQUIRE_EMBEDDING_ASSIGNMENT = True
MOSSFORMER_DISABLE_LOW_CONFIDENCE_ASR = True

OVERLAP_MARK_THRESHOLD_SECONDS = 0.05
OVERLAP_REVIEW_THRESHOLD_SECONDS = 0.05
OVERLAP_REQUIRE_DIFFERENT_SPEAKER = True
RUN_PYANNOTE_OSD = True
PYANNOTE_OSD_MODEL = "pyannote/overlapped-speech-detection"
PYANNOTE_OSD_MIN_DURATION_SECONDS = 0.05
PYANNOTE_OSD_FAIL_OPEN = True

# Giữ các biến cũ để cell/log không bị nhầm, nhưng stage 03 không dùng SepReformer nữa.
MIN_SEPREFORMER_OVERLAP_SECONDS = 1.0
MIN_SEPREFORMER_SEGMENT_SECONDS = 1.0

# Guard cho ASR ensemble: sửa các đoạn quá ngắn bị Whisper hallucinate.
ASR_QUALITY_GUARD = True
ASR_MICRO_SEGMENT_SECONDS = 0.5
ASR_SHORT_SEGMENT_SECONDS = 1.0
ASR_VI_AGREEMENT_THRESHOLD = 0.75

# ASR context padding: model nghe thêm biên trước/sau nhưng timestamp export vẫn giữ segment gốc.
ASR_CONTEXT_PAD_BEFORE_SECONDS = 0.25
ASR_CONTEXT_PAD_AFTER_SECONDS = 0.35

# ASRMoE chạy cả 3 model tiếng Việt: Whisper + PhoWhisper + ChunkFormer.
# Trên Kaggle 2xT4: Whisper đặt GPU0, PhoWhisper/ChunkFormer đặt GPU1.
ASR_MOE = True
WHISPER_DEVICE_INDEX = 0
VI_ASR_DEVICE_INDEX = 1
WHISPER_ARCH = "large-v3"
COMPUTE_TYPE = "float16"
ASR_THREADS = 4

# Initial prompt chỉ áp dụng cho Whisper. Giữ ngắn để giảm bias/hallucination.
USE_WHISPER_INITIAL_PROMPT = True
WHISPER_INITIAL_PROMPT = (
    "Podcast tiếng Việt tự nhiên, có thể xen từ tiếng Anh, tên app, brand và từ lóng. "
    "Từ đệm/backchannel thường gặp: ừ, ờ, à, dạ, vâng, đúng rồi, rồi, thì, là. "
    "Từ mượn/tên riêng: Facebook, Zalo, YouTube, TikTok, Instagram, Google, AI, livestream, podcast, content, deadline, feedback, booking, trend, viral, team, meeting, plank."
)

# PhoWhisper local/self-host: tải model về Kaggle và chạy local, không gọi HF Inference API.
WHISPER_HOTWORDS = "Facebook, Zalo, YouTube, TikTok, Instagram, Google, AI, livestream, podcast, content, deadline, feedback, booking, trend, viral, team, meeting, plank, vâng, dạ, đúng rồi"

PHOWHISPER_USE_HF_API = False
PHOWHISPER_API_MODEL = "vinai/PhoWhisper-large"
PHOWHISPER_API_PROVIDER = "hf-inference"
PHOWHISPER_API_TIMEOUT_SECONDS = 180

HF_SECRET_NAME = "HF_TOKEN"


In [ ]:

from pathlib import Path
import os
import shlex
import subprocess

LOG_DIR = Path(LOG_DIR_PATH)
for _dir in [INPUT_DIR, DIAR_DIR, MUSIC_DIR, OVERLAP_DIR, ASR_DIR, EXPORT_DIR, FINAL_DIR, EVAL_DIR, PREVIEW_DIR, LOG_DIR_PATH]:
    Path(_dir).mkdir(parents=True, exist_ok=True)


def _format_cmd(cmd):
    if isinstance(cmd, (list, tuple)):
        return " ".join(shlex.quote(str(part)) for part in cmd)
    return str(cmd)


def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])


def run_logged(cmd, log_name, cwd=None, env=None, shell=False, tail=20):
    log_path = LOG_DIR / log_name
    cwd = cwd or os.getcwd()
    print("Running:", _format_cmd(cmd))
    print("Log:", log_path)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print("Exit code:", proc.returncode)
    if tail:
        log_tail = tail_file(log_path, n=tail)
        if log_tail:
            print(f"--- last {tail} log lines ---")
            print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path


def export_audio_preview(audio_segment, out_path, seconds=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio_segment[: int(seconds * 1000)].export(out_path, format="wav")
    return out_path


## 1. Clone repo

In [ ]:

import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
repo_dir = Path("/kaggle/working/sommelier")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run_logged(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], "01_clone_repo.log", cwd="/kaggle/working", tail=20)
os.chdir(repo_dir / "podcast-pipeline")
print("cwd:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())


## 2. Cài dependencies từ internet

Bản standalone cài trực tiếp trong notebook này. Không cần private wheels dataset.


In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged(["apt-get", "update", "-y"], "02_apt_update.log", tail=10)
run_logged(["apt-get", "install", "-y", "ffmpeg", "git", "git-lfs"], "03_apt_install.log", tail=10)
run_logged(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "ninja"], "04_pip_base.log", tail=12)

req = Path("requirements.txt").read_text(encoding="utf-8")
filtered = [line for line in req.splitlines() if "nemo-toolkit[all]" not in line]
Path("requirements-kaggle.txt").write_text("\n".join(filtered) + "\n", encoding="utf-8")
run_logged(["python", "-m", "pip", "install", "-r", "requirements-kaggle.txt"], "05_pip_requirements.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "nemo-toolkit", "lightning", "pytorch-lightning"], "06_pip_uninstall_nemo.log", tail=8)
run_logged(["python", "-m", "pip", "install", "lightning==2.4.0", "pytorch-lightning==2.5.2"], "07_pip_lightning.log", tail=12)
run_logged(["python", "-m", "pip", "install", "nemo-toolkit[asr]==2.4.0"], "08_pip_nemo_asr.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], "09_pip_uninstall_torch.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
], "10_pip_torch_stack.log", tail=20)

run_logged(["python", "-m", "pip", "install", "pillow<12.0"], "11_pip_pillow.log", tail=8)
run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "torchmetrics==1.7.4"], "12_pip_torchmetrics.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "numpy==2.2.6", "numba==0.61.2", "llvmlite==0.44.0",
], "13_pip_numpy_numba.log", tail=12)

run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "clearvoice==0.1.2"], "14_pip_clearvoice.log", tail=20)

print("Dependency install logs saved in:", LOG_DIR)


## 3. Kiểm tra môi trường

In [ ]:
import importlib.metadata as importlib_metadata
import shutil
import subprocess
import numpy, numba, torch

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; kiểm tra torch.cuda bên dưới.")

print("nemo-toolkit:", importlib_metadata.version("nemo-toolkit"))
print("chunkformer:", importlib_metadata.version("chunkformer"))
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i))

import whisperx
print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")


## 4. Gắn Hugging Face token vào config

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")

## 5. Tìm audio input và chuẩn hóa audio

Output: `AUDIO_WAV = /kaggle/working/audio/full.wav`.

Chuẩn hóa về mono 16 kHz để các model dùng cùng format.

In [ ]:

from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

cmd = ["ffmpeg", "-hide_banner", "-y", "-i", AUDIO_IN]
if AUDIO_LIMIT_SECONDS:
    cmd += ["-t", str(AUDIO_LIMIT_SECONDS)]
cmd += ["-ac", "1", "-ar", "16000", AUDIO_WAV]
run_logged(cmd, "00_prepare_audio_ffmpeg.log", cwd="/kaggle/working", tail=15)


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)

preview_path = export_audio_preview(audio, Path(INPUT_DIR) / "preview_input_30s.wav", seconds=30)
print("Preview first 30s:", preview_path)
display(Audio(str(preview_path)))


## 6. Tải model phụ cho music clean và chuẩn bị MossFormer2 overlap separation

Bản này không clone SepReformer và không gọi HF audio-to-audio API ở stage 3. Stage 3 dùng package `clearvoice` và model local `MossFormer2_SS_16K`; model sẽ được ClearVoice tải về trong runtime Kaggle khi chạy lần đầu.


In [ ]:
from huggingface_hub import hf_hub_download

if RUN_DEMUCS:
    panns_path = hf_hub_download(
        repo_id="thelou1s/panns-inference",
        filename="Cnn14_mAP=0.431.pth",
        local_dir="/kaggle/working/sommelier/panns_data",
    )
    print("PANNs checkpoint:", panns_path)
else:
    print("RUN_DEMUCS=False, bỏ qua tải PANNs")

In [ ]:
import os
from pathlib import Path

print("Stage 03 mode: MossFormer2 self-host/local")
print("Model:", MOSSFORMER_MODEL_NAME)
print("Task:", MOSSFORMER_TASK)
print("Không clone/tải SepReformer local.")
print("Không gọi Hugging Face Inference API.")
print("Overlap mark threshold:", OVERLAP_MARK_THRESHOLD_SECONDS)
print("MossFormer min overlap:", MOSSFORMER_MIN_OVERLAP_SECONDS)
print("MossFormer context:", MOSSFORMER_CONTEXT_SECONDS)
print("Require different speaker:", OVERLAP_REQUIRE_DIFFERENT_SPEAKER)

Path(OVERLAP_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{OVERLAP_DIR}/separated_segments").mkdir(parents=True, exist_ok=True)
Path(f"{OVERLAP_DIR}/mossformer_windows").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working/sommelier/podcast-pipeline")


## 7. Trace VAD chunking

Bước này chỉ để xem VAD chia audio thành các chunk dài thế nào trước diarization. Đây không phải output speaker segment cuối cùng.

Output:
- `/kaggle/working/run_full/01_diarization/trace_vad_chunks.json`
- `/kaggle/working/run_full/01_diarization/vad_chunks/*.wav`

In [ ]:

import os
import json
import shutil
from pathlib import Path
import pandas as pd
from pydub import AudioSegment
from IPython.display import display

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

import stage_common
import main_original_ASR_MoE as pipeline

cfg = pipeline.load_cfg("config.json")
logger = pipeline.Logger.get_logger()
pipeline.cfg = cfg
pipeline.logger = logger

device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
device = pipeline.torch.device(device_name)
pipeline.device_name = device_name
pipeline.device = device
pipeline.vad = pipeline.silero_vad.SileroVAD(device=device)

sample_rate = int(cfg["entrypoint"]["SAMPLE_RATE"])
audio_info = stage_common.load_audio_info(AUDIO_WAV, sample_rate)
diar_chunks, temp_chunk_dir = pipeline.prepare_diarization_chunks(AUDIO_WAV, audio_info)

chunk_dir = Path(DIAR_DIR) / "vad_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

trace_chunks = []
for idx, chunk in enumerate(diar_chunks):
    src = Path(chunk["path"])
    dst = chunk_dir / f"chunk_{idx:03d}.wav"
    shutil.copy2(src, dst)
    duration = AudioSegment.from_file(dst).duration_seconds
    trace_chunks.append({
        "index": f"{idx:03d}",
        "path": str(dst),
        "offset": float(chunk["offset"]),
        "duration": float(duration),
        "start": float(chunk["offset"]),
        "end": float(chunk["offset"] + duration),
    })

if temp_chunk_dir:
    shutil.rmtree(temp_chunk_dir, ignore_errors=True)

stage_common.dump_json({
    "audio_path": AUDIO_WAV,
    "sample_rate": sample_rate,
    "chunks": trace_chunks,
    "metadata": {"stage": "vad_chunk_trace"},
}, Path(DIAR_DIR) / "trace_vad_chunks.json")

df_chunks = pd.DataFrame(trace_chunks)
print("VAD chunks:", len(df_chunks))
display(df_chunks.head(20))


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

if trace_chunks:
    first = trace_chunks[0]
    print(first)
    chunk_audio = AudioSegment.from_file(first["path"])
    preview_path = export_audio_preview(chunk_audio, Path(DIAR_DIR) / "preview_vad_chunk_0_30s.wav", seconds=30)
    print("Preview first 30s of chunk 0:", preview_path)
    display(Audio(str(preview_path)))


## 8. Stage 01 - Speaker diarization

Output: `/kaggle/working/run_full/01_diarization/diarization.json`.

Đây là bước Sortformer + speaker linking, tạo segment có `start`, `end`, `speaker`.

In [ ]:

import os
os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged([
    "python", "stage_01_diarize.py",
    "--input_audio", AUDIO_WAV,
    "--out", f"{DIAR_DIR}/diarization.json",
    "--merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--max_segment_duration", "30.0",
    "--same_speaker_merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--short_backchannel_seconds", str(SHORT_BACKCHANNEL_SECONDS),
    "--sortformer-pad-onset", "0.05",
    "--sortformer-pad-offset", "0.05",
], "18_stage_01_diarize.log", tail=25)


In [ ]:

import json
import pandas as pd
from IPython.display import display

with open(f"{DIAR_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)

diar_segments = diar["segments"]
df_diar = pd.DataFrame(diar_segments)
df_diar["dur"] = df_diar["end"].astype(float) - df_diar["start"].astype(float)

print("File:", f"{DIAR_DIR}/diarization.json")
print("Total segments:", len(df_diar))
print("Speakers:", sorted(df_diar["speaker"].unique()) if len(df_diar) else [])
print("Duration median:", df_diar["dur"].median() if len(df_diar) else 0)
print("Duration mean:", df_diar["dur"].mean() if len(df_diar) else 0)
print("< 1s:", int((df_diar["dur"] < 1).sum()) if len(df_diar) else 0)
print("< 2s:", int((df_diar["dur"] < 2).sum()) if len(df_diar) else 0)
print("< 3s:", int((df_diar["dur"] < 3).sum()) if len(df_diar) else 0)

display(df_diar[["index", "start", "end", "dur", "speaker"]].head(30))


## 9. Đóng gói kết quả Diarization

Nén thư mục kết quả Diarization thành ZIP để tiện tải về.

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

diar_dir = Path(DIAR_DIR)
zip_base = Path("/kaggle/working/diarization_results_original")

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(diar_dir.parent),
    base_dir=diar_dir.name,
)

print("Created:", zip_path)
display(FileLink(zip_path))